# PerGAZE ChenLSTMISP-S: subject embedding finetuning on Colab (one GPU)

Use **Runtime > Change runtime type > GPU**. Mount the same Google Drive used
by kaggle_chenlstm_colab.ipynb. This notebook reads the dataset and checkpoint
directly from Drive, without copying dataset files to /content.

Default K=5, seed=10, 3 supervised adaptation epochs. K samples are selected
independently for each unseen identity: ta:7/8/9, tp:7/8/9, air:JY/SC/YN.
TA user 7 and TP user 7 are separate people. K=5 selects 45 support scanpaths.
Only new rows of subject_embedding.weight are updated; ResNet, ConvLSTM,
COCO object heads, VQA head, old subject rows and all buffers remain frozen.

Every epoch evaluates test_unseen and saves the best ScanMatch harmonic mean.
As requested, test_unseen selects the best epoch; this is model selection on
that set, not an untouched held-out test estimate. No text embedding archive
or SentenceTransformer is required for ChenLSTM.


In [ ]:
from google.colab import drive
drive.mount("/content/drive")


In [ ]:
import subprocess
import sys
from pathlib import Path

CODE = Path("/content/IndividualScanpath")
if (CODE / ".git").is_dir():
    subprocess.run(["git", "-C", str(CODE), "pull", "--ff-only"], check=True)
else:
    subprocess.run(["git", "clone", "--depth", "1",
                    "https://github.com/ntnghia06/IndividualScanpath.git", str(CODE)], check=True)
SRC = CODE / "PerGAZE/ChenLSTMISP-S/src"
assert (SRC / "train.py").is_file(), "Update repository for ChenLSTMISP-S"


In [ ]:
%pip install -q "numpy>=1.24" "Pillow>=9.1" "scipy>=1.10" "tqdm>=4.65" multimatch-gaze scikit-image


In [ ]:
import torch

assert torch.cuda.is_available(), "Select GPU in Runtime > Change runtime type"
GPU_IDS = [0]
print("PyTorch:", torch.__version__)
print("GPU:", torch.cuda.get_device_name(0))
print("GPU memory GiB:", torch.cuda.get_device_properties(0).total_memory / 2**30)


## Paths and hyperparameters

The default base checkpoint is the best output of kaggle_chenlstm_colab.ipynb:

```text
MyDrive/PerGAZE/
  dataset/
    support.json
    test_unseen.json
    images/
    attention_reasoning/
  runs/pergaze_chenlstm_colab_1gpu/checkpoints/best.pth
```

Edit DRIVE_ROOT or BASE_RUN if your training run used different paths. The base
checkpoint must have separate TP/TA subjects, 18 COCO object branches and the VQA
branch. Older pooled-observer or shared-head checkpoints are incompatible.
max_length, embedding dimension and other architecture settings come from the
checkpoint. K, support seed and adaptation learning rate are adjustable below.

Batch defaults are 1 for this ConvLSTM on one GPU. SEED=10 controls support
sampling and adaptation; the base training seed may be 1. RUN is on Drive so
last/best checkpoints and the single report.json persist after each epoch.


In [ ]:
K = 5
SEED = 10
EPOCHS = 3
LEARNING_RATE = 1e-3
TRAIN_BATCH = 1
TEST_BATCH = 1
WORKERS = 2
RESUME = False

DRIVE_ROOT = Path("/content/drive/MyDrive/PerGAZE")
DATA = DRIVE_ROOT / "dataset"
BASE_RUN = DRIVE_ROOT / "runs/pergaze_chenlstm_colab_1gpu"
CHECKPOINT = BASE_RUN / "checkpoints/best.pth"
RUN = DRIVE_ROOT / "runs" / f"chenlstm_s_k{K}_seed{SEED}_colab_1gpu"
LOCAL_ROOT = Path("/content/pergaze_chenlstm_s_work")  # Logs/optional ZIP only.

for name in ("support.json", "test_unseen.json", "images", "attention_reasoning"):
    assert (DATA / name).exists(), f"Missing: {DATA / name}"
assert CHECKPOINT.is_file(), f"Missing trained ChenLSTM base checkpoint: {CHECKPOINT}"
assert min(K, EPOCHS, TRAIN_BATCH, TEST_BATCH) >= 1 and LEARNING_RATE > 0 and WORKERS >= 0
RUN.mkdir(parents=True, exist_ok=True)
if RESUME:
    assert (RUN / "checkpoints/checkpoint.pth").is_file(), f"No adaptation checkpoint at {RUN}"
print("Direct Drive dataset:", DATA)
print("Base checkpoint:", CHECKPOINT)
print("Adaptation output:", RUN)


In [ ]:
import shutil

COMMON = [
    "--checkpoint", str(CHECKPOINT),
    "--support_file", str(DATA / "support.json"),
    "--test_file", str(DATA / "test_unseen.json"),
    "--img_dir", str(DATA / "images"),
    "--att_dir", str(DATA / "attention_reasoning"),
    "--k", str(K), "--seed", str(SEED), "--epoch", str(EPOCHS),
    "--lr", str(LEARNING_RATE),
    "--device", "cuda:0", "--gpu_ids", "0",
]

def run_script(script, *arguments):
    import codecs
    import time

    command = [sys.executable, "-u", str(SRC / script), *COMMON, *arguments]
    local_logs = LOCAL_ROOT / "logs"
    local_logs.mkdir(parents=True, exist_ok=True)
    log_path = local_logs / f"{Path(script).stem}_{time.time_ns()}.log"
    drive_log = RUN / "logs" / log_path.name
    decoder = codecs.getincrementaldecoder("utf-8")(errors="replace")
    tail = bytearray()
    process = None
    return_code = None
    print("Full subprocess log:", log_path, flush=True)
    try:
        with log_path.open("wb") as log:
            log.write(("Command: " + repr(command) + "\n").encode("utf-8"))
            process = subprocess.Popen(command, stdout=subprocess.PIPE,
                                       stderr=subprocess.STDOUT)
            try:
                while True:
                    chunk = process.stdout.read1(8192)
                    if not chunk:
                        break
                    log.write(chunk)
                    log.flush()
                    tail.extend(chunk)
                    if len(tail) > 65536:
                        del tail[:-65536]
                    sys.stdout.write(decoder.decode(chunk))
                    sys.stdout.flush()
                sys.stdout.write(decoder.decode(b"", final=True))
                return_code = process.wait()
            finally:
                if process.poll() is None:
                    process.terminate()
                    try:
                        process.wait(timeout=10)
                    except subprocess.TimeoutExpired:
                        process.kill()
                        process.wait()
                process.stdout.close()
    finally:
        if log_path.is_file():
            try:
                drive_log.parent.mkdir(parents=True, exist_ok=True)
                shutil.copy2(log_path, drive_log)
                print("Log copied to Drive:", drive_log, flush=True)
            except OSError as error:
                print("Could not copy log to Drive; local log is retained:", error, flush=True)
    if return_code:
        lines = tail.decode("utf-8", errors="replace").replace("\r", "\n").splitlines()
        last_lines = "\n".join(lines[-40:])
        hint = ""
        if "out of memory" in last_lines.lower():
            hint = "\nMemory error: for a NEW run try TRAIN_BATCH=1, TEST_BATCH=1. Resume requires the saved train batch."
        elif "checkpoint exists" in last_lines.lower():
            hint = "\nSet RESUME=True to continue that run, or choose a new RUN directory."
        raise RuntimeError(f"{script} exited with code {return_code}. Full log: {log_path}"
                           f"\nLast subprocess output:\n{last_lines}{hint}")


## Finetune and evaluate

Baseline evaluation runs first, then support finetuning and full unseen eval
for each epoch. Progress bars show train, eval and retrieval. ResNet weights
are loaded from the trained checkpoint, so no pretrained download is needed.
Only subject embedding rows are optimized. The full subprocess output is shown
and logged locally, then copied to RUN/logs on Drive when the script exits.


In [ ]:
arguments = ["--log_root", str(RUN), "--workers", str(WORKERS),
             "--batch", str(TRAIN_BATCH), "--test_batch", str(TEST_BATCH)]
if RESUME:
    arguments.append("--resume")
run_script("train.py", *arguments)


In [ ]:
import json
report = json.loads((RUN / "report.json").read_text(encoding="utf-8"))
print("Selected support samples:", report["support_samples"])
print("Adapted subjects:", report["adaptive_subjects"])
print("Best epoch:", report["best_epoch"], "score:", report["best_score"])
print(json.dumps(report["best_metrics"]["overall"], indent=2))
print(json.dumps(report["best_metrics"].get("retrieval", {}).get("by_condition", {}), indent=2))
print("Full report:", RUN / "report.json")


## Optional re-evaluation

Uncomment the next cell to evaluate adapted best.pth again. The later checkpoint
argument replaces the base checkpoint in COMMON. Results are appended to the
reevaluation section in the same report.json.


In [ ]:
# run_script("test.py", "--checkpoint", str(RUN / "checkpoints/best.pth"),
#            "--log_root", str(RUN), "--workers", str(WORKERS),
#            "--test_batch", str(TEST_BATCH))


## Resume and outputs

After session interruption, mount Drive, rerun setup with RESUME=True and the
same RUN, K, seed, learning rate, train/eval batches and input files. Keep the
original base checkpoint. A partially completed epoch restarts from the last
saved completed epoch. Use a different RUN for a new experiment.

Outputs on Drive: checkpoints/checkpoint.pth (last), checkpoints/best.pth and
report.json with selected support IDs, baseline, epoch history, best metrics,
predictions and retrieval matrices. The report also records config and input
hashes. The next cell optionally creates a ZIP for download.


In [ ]:
print("Outputs saved on Drive:", RUN)
# LOCAL_ROOT.mkdir(parents=True, exist_ok=True)
# archive = shutil.make_archive(str(LOCAL_ROOT / f"chenlstm_s_k{K}_seed{SEED}"),
#                              "zip", root_dir=RUN.parent, base_dir=RUN.name)
# from google.colab import files
# files.download(archive)
